# LeetCode 2152 — Minimum Number of Lines to Cover Points bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-lines-to-cover-points/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Dynamic Programming, Backtracking, Bit Manipulation, Geometry, Bitmask

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def minimumLines(self, points: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Kiểm tra **ba điểm thẳng hàng** bằng **tích chéo** (cross product) với số nguyên — không dùng hệ số góc dạng số thực.
2. Biểu diễn "một đường thẳng" bằng **bitmask** các điểm nằm trên nó; chỉ cần xét các đường **tối đa** (đi qua ít nhất 2 điểm và chứa mọi điểm thẳng hàng với chúng) cộng các đường "chỉ một điểm".
3. Giải bài phủ tập hợp (set cover) nhỏ bằng **quy hoạch động trên tập con** `dp[mask]`, với mẹo **luôn phủ điểm chưa phủ có chỉ số nhỏ nhất** để cắt nhánh.
4. Thấy tham lam "chọn đường phủ nhiều điểm mới nhất" **sai** (phản ví dụ cụ thể).
5. Biết thêm cách tìm theo **số đường k tăng dần** (iterative deepening) và cách chuẩn hóa đường thẳng bằng bộ `(a, b, c)` tối giản.

## 2. Đề bài & ví dụ

Cho mảng `points`, `points[i] = [x_i, y_i]` là các điểm (**phân biệt**) trên mặt phẳng. Ta vẽ thêm các **đường thẳng** sao cho **mọi điểm** nằm trên ít nhất một đường. Trả về **số đường thẳng ít nhất**.

**Ví dụ 1:** `points = [[0,1],[2,3],[4,5],[4,3]]` → `2`. Một cách: đường qua `(0,1)` và `(4,5)` (cũng đi qua `(2,3)`), đường thứ hai qua `(4,3)` (có thể nối `(2,3)`–`(4,3)`).

**Ví dụ 2:** `points = [[0,2],[-2,-2],[1,4]]` → `1` (ba điểm thẳng hàng trên `y = 2x + 2`).

**Ràng buộc:** `1 ≤ points.length ≤ 10`, `−100 ≤ x_i, y_i ≤ 100`, các điểm phân biệt.

Nhận xét ngay: mỗi đường phủ được ít nhất 2 điểm (nếu còn ≥ 2 điểm), nên đáp án ≤ `⌈n/2⌉ ≤ 5`.

In [1]:
from typing import List
from functools import lru_cache
from itertools import combinations
from math import gcd, inf
import random, time

examples = [([[0, 1], [2, 3], [4, 5], [4, 3]], 2), ([[0, 2], [-2, -2], [1, 4]], 1)]
for pts, exp in examples:
    print(pts, "-> kỳ vọng", exp)

[[0, 1], [2, 3], [4, 5], [4, 3]] -> kỳ vọng 2
[[0, 2], [-2, -2], [1, 4]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Tích chéo — kiểm tra thẳng hàng bằng số nguyên

Ba điểm `A, B, C` thẳng hàng ⇔ `(B − A) × (C − A) = 0`, tức `(xB − xA)(yC − yA) − (xC − xA)(yB − yA) = 0`. Chỉ dùng phép nhân/trừ số nguyên ⇒ **chính xác tuyệt đối**, không chia cho 0 với đường thẳng đứng.

In [2]:
def collinear(a, b, c):
    return (b[0] - a[0]) * (c[1] - a[1]) == (c[0] - a[0]) * (b[1] - a[1])

print(collinear([0, 2], [-2, -2], [1, 4]), collinear([0, 1], [2, 3], [4, 3]))
print("đường thẳng đứng:", collinear([1, 0], [1, 5], [1, -7]))

True False
đường thẳng đứng: True


So với dùng hệ số góc `(yB − yA)/(xB − xA)`: chia cho 0 với đường đứng và sai số `float` (ví dụ `1/3` không biểu diễn chính xác).

In [3]:
print(0.1 * 3 == 0.3, "| tích chéo chỉ dùng số nguyên nên không có vấn đề này")

False | tích chéo chỉ dùng số nguyên nên không có vấn đề này


### 3.2 Bitmask cho tập điểm

Với `n ≤ 10`, mỗi tập điểm là một số nguyên `< 2¹⁰ = 1024`: bit `i` bật ⇔ điểm `i` thuộc tập. Hợp: `a | b`; điểm `i` có trong `m`: `m >> i & 1`; tập đầy đủ: `(1 << n) − 1`.

In [4]:
m = 0b0101
print(bin(m), "có điểm 0?", m >> 0 & 1, "có điểm 1?", m >> 1 & 1, "| hợp với 0b0010 =", bin(m | 0b0010), "| full(4) =", bin((1 << 4) - 1))

0b101 có điểm 0? 1 có điểm 1? 0 | hợp với 0b0010 = 0b111 | full(4) = 0b1111


### 3.3 Tìm bit 0 thấp nhất (điểm chưa phủ có chỉ số nhỏ nhất)

`~m & (m + 1)` là bit 0 thấp nhất của `m` (dạng lũy thừa 2); `.bit_length() − 1` ra chỉ số.

In [5]:
for m in [0b0000, 0b0111, 0b1011]:
    low = ~m & (m + 1)
    print(bin(m), "-> bit 0 thấp nhất:", bin(low), "chỉ số", low.bit_length() - 1)

0b0 -> bit 0 thấp nhất: 0b1 chỉ số 0
0b111 -> bit 0 thấp nhất: 0b1000 chỉ số 3
0b1011 -> bit 0 thấp nhất: 0b100 chỉ số 2


### 3.4 Đường thẳng dạng chuẩn `(a, b, c)` với `ax + by = c`

Qua hai điểm `P, Q`: `a = yQ − yP`, `b = xP − xQ`, `c = a·xP + b·yP`. Chia cho `gcd(|a|, |b|, |c|)` và cố định dấu ⇒ mọi cặp điểm trên cùng một đường cho **cùng bộ**. Hữu ích để gom nhóm bằng `dict`.

In [6]:
def line_key(p, q):
    a = q[1] - p[1]; b = p[0] - q[0]; c = a * p[0] + b * p[1]
    g = gcd(gcd(abs(a), abs(b)), abs(c)) or 1
    a, b, c = a // g, b // g, c // g
    if a < 0 or (a == 0 and b < 0): a, b, c = -a, -b, -c
    return (a, b, c)

print(line_key([0, 2], [-2, -2]), line_key([1, 4], [0, 2]), line_key([-2, -2], [1, 4]))

(2, -1, -2) (2, -1, -2) (2, -1, -2)


## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — chỉ cần các đường "tối đa".** Một đường thẳng trong lời giải hoặc đi qua ≥ 2 điểm — khi đó nó được xác định bởi một **cặp** điểm và tự động phủ **mọi** điểm thẳng hàng với cặp đó; hoặc chỉ đi qua 1 điểm. Vậy tập ứng viên gồm ≤ `C(n,2) + n` đường (dạng bitmask).

**Quan sát 2 — thứ tự chọn đường không quan trọng**, chỉ quan trọng **tập** đã phủ ⇒ trạng thái DP là `mask`.

**Quan sát 3 — luôn xử lý điểm chưa phủ nhỏ nhất `i`.** Trong lời giải tối ưu, **có** một đường phủ `i`; ta có thể chọn nó trước. Do đó từ `mask` chỉ cần thử các đường đi qua `i` (≤ `n` đường), không cần thử mọi đường.

Tính tay ví dụ 1: điểm `0:(0,1), 1:(2,3), 2:(4,5), 3:(4,3)`.

| cặp | các điểm thẳng hàng | mask |
|---|---|---|
| (0,1) | 0, 1, 2 | 0b0111 |
| (0,3) | 0, 3 | 0b1001 |
| (1,3) | 1, 3 | 0b1010 |
| (2,3) | 2, 3 (x = 4) | 0b1100 |

Chọn `0b0111` (phủ 0, 1, 2) rồi một đường bất kỳ qua điểm 3 ⇒ 2 đường. Một đường là không thể (4 điểm không thẳng hàng) ⇒ đáp án 2.

In [7]:
def line_masks(points):
    n = len(points); masks = set()
    for i in range(n):
        masks.add(1 << i)
        for j in range(i + 1, n):
            m = 0
            for k in range(n):
                if collinear(points[i], points[j], points[k]):
                    m |= 1 << k
            masks.add(m)
    return masks

pts = examples[0][0]
for m in sorted(line_masks(pts), key=lambda x: (-bin(x).count("1"), x)):
    print(f"{m:04b} ->", [pts[k] for k in range(len(pts)) if m >> k & 1])

0111 -> [[0, 1], [2, 3], [4, 5]]
1001 -> [[0, 1], [4, 3]]
1010 -> [[2, 3], [4, 3]]
1100 -> [[4, 5], [4, 3]]
0001 -> [[0, 1]]
0010 -> [[2, 3]]
0100 -> [[4, 5]]
1000 -> [[4, 3]]


## 5. Cách 1 — Brute Force / baseline: thử mọi tổ hợp k đường, k tăng dần

**Ý tưởng:** liệt kê các đường ứng viên (bitmask, đã khử trùng), rồi với `k = 1, 2, …` thử mọi tổ hợp `k` đường; tổ hợp đầu tiên có hợp = tập đầy đủ cho đáp án.

In [8]:
def brute_combos(points):
    n = len(points); full = (1 << n) - 1
    L = sorted(line_masks(points))
    for k in range(1, n + 1):
        for combo in combinations(L, k):
            u = 0
            for m in combo: u |= m
            if u == full:
                return k
    return n

for pts, exp in examples:
    got = brute_combos(pts)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp:** số đường ứng viên `L ≤ 45 + 10 = 55`; đáp án `k ≤ 5` ⇒ worst ≈ `C(55, 5) ≈ 3,5·10⁶` tổ hợp — chạy được với n = 10 nhưng chậm trong Python (vài giây trong worst case). Nếu ràng buộc nới lên n = 16 thì không còn khả thi.

**Nút thắt:** nhiều tổ hợp chứa các đường dư thừa hoặc chỉ khác thứ tự; và ta không tận dụng việc "phải phủ điểm i nào đó".

### 5.1 Baseline thứ hai: quay lui gán từng điểm vào một nhóm thẳng hàng

Đi lần lượt từng điểm, gán vào một nhóm đang có (nếu vẫn thẳng hàng) hoặc mở nhóm mới; cắt nhánh khi số nhóm ≥ kết quả tốt nhất. Đây là cách nghĩ "phân hoạch".

In [9]:
def backtrack_groups(points):
    n = len(points); best = [n]; groups = []
    def fits(g, p):
        if len(g) < 2: return True
        return collinear(points[g[0]], points[g[1]], p)
    def go(i):
        if len(groups) >= best[0]: return
        if i == n:
            best[0] = len(groups); return
        for g in groups:
            if fits(g, points[i]):
                g.append(i); go(i + 1); g.pop()
        groups.append([i]); go(i + 1); groups.pop()
    go(0)
    return best[0]

for pts, exp in examples:
    print(backtrack_groups(pts), "OK" if backtrack_groups(pts) == exp else "SAI")

2 OK
1 OK


Mỗi điểm chỉ thuộc **một** nhóm (phân hoạch) — không mất tổng quát vì nếu một điểm nằm trên nhiều đường, ta chỉ cần "giao" nó cho một đường.

## 6. Ý tưởng thuật toán / bước đột phá

**DP trên tập con:** `dp[mask]` = số đường ít nhất để phủ **thêm** mọi điểm ngoài `mask` (hoặc tương đương: số đường ít nhất để đạt tập phủ `mask` từ rỗng).

Quy tắc chuyển (dạng "từ mask đi tiếp"):
- **Nếu** `mask` đã đầy **thì** xong.
- Gọi `i` = điểm chưa phủ có chỉ số nhỏ nhất.
- **Với mỗi** đường ứng viên `L` đi qua `i` (đường qua `i` và `j` với `j ≠ i` — và đường chỉ có `i`): `dp[mask | L] = min(dp[mask | L], dp[mask] + 1)`.

**Có cần cho `j` là điểm đã phủ không?** Không bắt buộc. Nếu đường `L` qua `i` còn chứa một điểm **chưa phủ** `k ≠ i` thì `L = line(i, k)` — đã được xét khi ghép `i` với `k`. Nếu `L` không chứa điểm chưa phủ nào khác ngoài `i` thì về hiệu quả nó tương đương đường một điểm `{i}`. Vậy chỉ cần `j` chạy trên các điểm **chưa phủ** (một tối ưu nhỏ); cho `j` chạy trên mọi điểm cũng đúng (đơn giản hơn khi tiền xử lý một lần). Mục 13 kiểm chứng hai cách cho cùng kết quả.

**Bất biến:** khi xử lý `mask` theo thứ tự tăng dần, `dp[mask]` đã là giá trị cuối cùng — vì mọi chuyển trạng thái đều đi từ `mask` sang `mask | L ⊇ mask` (số lớn hơn hoặc bằng; bằng chỉ khi `L ⊆ mask`, nhưng `L` luôn chứa `i ∉ mask` nên luôn lớn hơn).

**Độ phức tạp:** `2ⁿ` trạng thái × `n` đường qua `i` = `O(2ⁿ · n)` sau khi tiền xử lý `O(n³)` các mask đường.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP lặp trên mask (bottom-up), luôn phủ điểm nhỏ nhất chưa phủ

In [10]:
def dp_bottom_up(points):
    n = len(points); full = (1 << n) - 1
    through = [[1 << i] for i in range(n)]          # các đường đi qua điểm i
    for i in range(n):
        for j in range(n):
            if j == i: continue
            m = 0
            for k in range(n):
                if collinear(points[i], points[j], points[k]):
                    m |= 1 << k
            through[i].append(m)
    dp = [inf] * (1 << n); dp[0] = 0
    for mask in range(full):
        if dp[mask] == inf: continue
        low = ~mask & (mask + 1); i = low.bit_length() - 1
        for L in through[i]:
            nm = mask | L
            if dp[mask] + 1 < dp[nm]:
                dp[nm] = dp[mask] + 1
    return dp[full]

for pts, exp in examples:
    got = dp_bottom_up(pts)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp cách 2:** tiền xử lý O(n³) = 1000; DP O(2ⁿ · n) ≈ 10⁴ phép với n = 10. Bộ nhớ O(2ⁿ).

### 7.2 Cách 3 — Đệ quy có nhớ (top-down) trên tập đã phủ

`f(mask)` = số đường ít nhất để phủ nốt phần còn lại. Chỉ các mask **thực sự đạt tới** mới được tính.

In [11]:
def dp_top_down(points):
    n = len(points); full = (1 << n) - 1
    line = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                line[i][j] = sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
    @lru_cache(None)
    def f(mask):
        if mask == full: return 0
        i = (~mask & (mask + 1)).bit_length() - 1
        best = 1 + f(mask | 1 << i)                  # đường chỉ phủ i (có thể vẽ qua i tùy ý)
        for j in range(n):
            if j != i:
                best = min(best, 1 + f(mask | line[i][j]))
        return best
    return f(0)

for pts, exp in examples:
    got = dp_top_down(pts)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


Độ sâu đệ quy ≤ số đường ≤ n — an toàn.

**Độ phức tạp cách 3:** như cách 2 nhưng thường ít trạng thái hơn (chỉ mask đạt được).

### 7.3 Cách 4 — Gom đường bằng khóa chuẩn `(a, b, c)` + BFS trên mask

Dùng `line_key` gom mọi cặp điểm cùng đường vào một `dict` ⇒ danh sách đường **khác nhau** (mỗi đường một mask). Sau đó BFS từ mask rỗng: mỗi bước thêm một đường; lần đầu chạm `full` là đáp án (BFS cho số bước ít nhất).

In [12]:
from collections import deque, defaultdict
def bfs_lines(points):
    n = len(points); full = (1 << n) - 1
    groups = defaultdict(int)
    for i in range(n):
        for j in range(i + 1, n):
            key = line_key(points[i], points[j])
            groups[key] |= (1 << i) | (1 << j)
    L = set(groups.values()) | {1 << i for i in range(n)}
    dist = {0: 0}; q = deque([0])
    while q:
        m = q.popleft()
        if m == full: return dist[m]
        i = (~m & (m + 1)).bit_length() - 1
        for l in L:
            if l >> i & 1:
                nm = m | l
                if nm not in dist:
                    dist[nm] = dist[m] + 1; q.append(nm)
    return 0

for pts, exp in examples:
    got = bfs_lines(pts)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp cách 4:** O(n² log C) để chuẩn hóa (gcd) + O(2ⁿ · |L|) BFS. Ưu điểm: khóa `(a, b, c)` là kỹ thuật chuẩn của LC 149 (Max Points on a Line).

### 7.4 Cách 5 — Tìm kiếm theo k tăng dần với cắt nhánh (iterative deepening)

Thử "có phủ được bằng `k` đường không?" với `k = 1, 2, …`: quay lui, luôn phủ điểm chưa phủ nhỏ nhất; cắt khi số đường còn lại `r` không đủ: mỗi đường phủ tối đa `maxLine` điểm ⇒ cần `r · maxLine ≥ số điểm còn lại`.

In [13]:
def iterative_deepening(points):
    n = len(points); full = (1 << n) - 1
    through = [sorted({1 << i} | {sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                                  for j in range(n) if j != i}, key=lambda m: -bin(m).count("1")) for i in range(n)]
    max_line = max(bin(m).count("1") for t in through for m in t)
    def can(mask, r):
        if mask == full: return True
        if r == 0 or r * max_line < n - bin(mask).count("1"): return False
        i = (~mask & (mask + 1)).bit_length() - 1
        return any(can(mask | L, r - 1) for L in through[i])
    k = 1
    while not can(0, k): k += 1
    return k

for pts, exp in examples:
    got = iterative_deepening(pts)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


## 8. TRACE từng bước

Trace cách 2 (DP bottom-up) trên ví dụ 1, in mọi trạng thái đạt được và chuyển trạng thái.

In [14]:
def trace_dp(points):
    n = len(points); full = (1 << n) - 1
    through = [sorted({1 << i} | {sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                                  for j in range(n) if j != i}) for i in range(n)]
    dp = [inf] * (1 << n); dp[0] = 0
    for mask in range(full):
        if dp[mask] == inf: continue
        i = (~mask & (mask + 1)).bit_length() - 1
        moves = []
        for L in through[i]:
            nm = mask | L
            if dp[mask] + 1 < dp[nm]: dp[nm] = dp[mask] + 1
            moves.append(f"{L:0{n}b}->{nm:0{n}b}")
        print(f"mask={mask:0{n}b} (dp={dp[mask]}), điểm nhỏ nhất chưa phủ = {i}: {', '.join(moves)}")
    print("dp[full] =", dp[full])

trace_dp(examples[0][0])

mask=0000 (dp=0), điểm nhỏ nhất chưa phủ = 0: 0001->0001, 0111->0111, 1001->1001
mask=0001 (dp=1), điểm nhỏ nhất chưa phủ = 1: 0010->0011, 0111->0111, 1010->1011
mask=0011 (dp=2), điểm nhỏ nhất chưa phủ = 2: 0100->0111, 0111->0111, 1100->1111
mask=0111 (dp=1), điểm nhỏ nhất chưa phủ = 3: 1000->1111, 1001->1111, 1010->1111, 1100->1111
mask=1001 (dp=1), điểm nhỏ nhất chưa phủ = 1: 0010->1011, 0111->1111, 1010->1011
mask=1011 (dp=2), điểm nhỏ nhất chưa phủ = 2: 0100->1111, 0111->1111, 1100->1111
dp[full] = 2


Đọc kết quả (bit phải nhất là điểm 0): từ `0000` chỉ thử các đường qua điểm 0 (`0001`, `0111`, `1001`); từ `0111` (đã phủ 0, 1, 2) chỉ thử đường qua điểm 3 ⇒ `1111` với `dp = 2`.

Trace cách 5 (k tăng dần) trên phản ví dụ của tham lam (xem mục 13):

In [15]:
P_bad = [[0, 2], [1, 0], [1, 1], [1, 3], [2, 0], [2, 1], [3, 1], [3, 2]]
def trace_deepening(points):
    n = len(points); full = (1 << n) - 1
    through = [sorted({1 << i} | {sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                                  for j in range(n) if j != i}, key=lambda m: -bin(m).count("1")) for i in range(n)]
    calls = [0]
    def can(mask, r):
        calls[0] += 1
        if mask == full: return True
        if r == 0: return False
        i = (~mask & (mask + 1)).bit_length() - 1
        return any(can(mask | L, r - 1) for L in through[i])
    k = 1
    while True:
        calls[0] = 0; ok = can(0, k)
        print(f"k={k}: {'phủ được' if ok else 'không'} ({calls[0]} lời gọi)")
        if ok: return k
        k += 1

trace_deepening(P_bad)

k=1: không (8 lời gọi)
k=2: không (49 lời gọi)
k=3: phủ được (8 lời gọi)


3

## 9. Code nộp LeetCode

In [16]:
from typing import List
from math import inf

class Solution:
    def minimumLines(self, points: List[List[int]]) -> int:
        n = len(points)
        full = (1 << n) - 1

        def collinear(a, b, c):
            return (b[0] - a[0]) * (c[1] - a[1]) == (c[0] - a[0]) * (b[1] - a[1])

        # through[i]: các đường (dạng bitmask) đi qua điểm i
        through = []
        for i in range(n):
            cand = {1 << i}
            for j in range(n):
                if j != i:
                    m = 0
                    for k in range(n):
                        if collinear(points[i], points[j], points[k]):
                            m |= 1 << k
                    cand.add(m)
            through.append(cand)

        dp = [inf] * (1 << n)
        dp[0] = 0
        for mask in range(full):
            if dp[mask] == inf:
                continue
            i = (~mask & (mask + 1)).bit_length() - 1      # điểm chưa phủ nhỏ nhất
            for line in through[i]:
                nm = mask | line
                if dp[mask] + 1 < dp[nm]:
                    dp[nm] = dp[mask] + 1
        return dp[full]

In [17]:
sol = Solution()
for pts, exp in examples:
    got = sol.minimumLines(pts)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
1 | kỳ vọng 1 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đường tối đa):** tồn tại lời giải tối ưu mà mỗi đường là một **đường ứng viên** — tức là, hoặc là đường qua hai điểm `i, j` (khi đó nó phủ đúng tập `line(i, j)` gồm mọi điểm thẳng hàng với `i, j`), hoặc chỉ phủ một điểm. Lý do: một đường bất kỳ phủ tập `S`; nếu `|S| ≥ 2` thì đường đó chính là đường qua hai điểm bất kỳ của `S` và `S = line(i, j)`; nếu `|S| = 1` thì là đường một điểm.

**Bổ đề 2 (điểm nhỏ nhất):** xét trạng thái `mask` và điểm chưa phủ nhỏ nhất `i`. Mọi lời giải hoàn chỉnh từ `mask` phải có một đường phủ `i`; đường đó (theo Bổ đề 1) là `line(i, j)` với một `j ≠ i` nào đó, hoặc đường đơn `{i}`. Chọn nó **trước** không làm thay đổi tổng số đường. Vậy `opt(mask) = 1 + min_{L ∈ through[i]} opt(mask | L)`.

**DP đúng:** cách 2 tính đúng hệ thức trên theo thứ tự mask tăng (mỗi chuyển trạng thái làm mask tăng nghiêm ngặt vì `L ∋ i ∉ mask`). ∎

**Lưu ý quan trọng:** mask của đường phải là đường **tối đa** `line(i, j)` (mọi điểm thẳng hàng với `i, j`), không phải chỉ `{i, j}` — xem Lỗi 3. Còn việc `j` chạy trên mọi điểm hay chỉ điểm chưa phủ thì tương đương (mục 6).

## 11. Phân tích Time / Space Complexity

`n ≤ 10`.

| Cách | Time | Space |
|---|---|---|
| 1. Tổ hợp k đường | O(C(L, k)·k), L ≤ n(n−1)/2 + n | O(L) |
| 1b. Quay lui phân hoạch | mũ (cắt nhánh tốt) | O(n) |
| 2. DP bottom-up | O(n³ + 2ⁿ·n) | O(2ⁿ) |
| 3. DP top-down | ≤ O(n³ + 2ⁿ·n) | O(2ⁿ) |
| 4. Khóa (a,b,c) + BFS | O(n² log C + 2ⁿ·L) | O(2ⁿ) |
| 5. k tăng dần + cắt nhánh | mũ theo k, rất nhanh với k ≤ 5 | O(n) |

In [18]:
rng = random.Random(2152)
pts_list = []
for _ in range(30):
    s = set()
    while len(s) < 10: s.add((rng.randint(-3, 3), rng.randint(-3, 3)))
    pts_list.append([list(p) for p in s])
for name, f in [("brute_combos", brute_combos), ("backtrack_groups", backtrack_groups), ("dp_bottom_up", dp_bottom_up),
                ("dp_top_down", dp_top_down), ("bfs_lines", bfs_lines), ("iterative_deepening", iterative_deepening),
                ("Solution", Solution().minimumLines)]:
    t = time.perf_counter(); r = [f(p) for p in pts_list]; dt = time.perf_counter() - t
    print(f"{name:<20} 30 bộ n=10: {dt*1000:6.0f} ms, đáp án đầu: {r[:8]}")

brute_combos         30 bộ n=10:    369 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
backtrack_groups     30 bộ n=10:      3 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
dp_bottom_up         30 bộ n=10:      6 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
dp_top_down          30 bộ n=10:      8 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
bfs_lines            30 bộ n=10:      9 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
iterative_deepening  30 bộ n=10:      6 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]
Solution             30 bộ n=10:      6 ms, đáp án đầu: [4, 5, 4, 4, 4, 4, 4, 4]


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [19]:
ALL = {"brute_combos": brute_combos, "backtrack_groups": backtrack_groups, "dp_bottom_up": dp_bottom_up,
       "dp_top_down": dp_top_down, "bfs_lines": bfs_lines, "iterative_deepening": iterative_deepening,
       "Solution": Solution().minimumLines}
edge = examples + [
    ([[5, 5]], 1),                                         # một điểm
    ([[0, 0], [1, 1]], 1),                                 # hai điểm luôn 1 đường
    ([[0, 0], [1, 0], [0, 1]], 2),                         # tam giác
    ([[0, 0], [0, 1], [0, 2], [0, 3]], 1),                 # đường đứng
    ([[0, 0], [1, 1], [2, 2], [0, 2], [2, 0]], 2),         # hai đường chéo của hình vuông 3x3 (5 điểm)
    ([[x, y] for x in range(3) for y in range(3)], 3),     # lưới 3x3: 3 hàng
    ([[0, 0], [1, 2], [2, 4], [3, 1], [4, 3], [5, 5]], 2), # y = 2x và (3,1),(4,3),(5,5) trên y = 2x - 5
    (P_bad, 3),
]
for pts, exp in edge:
    for name, f in ALL.items():
        assert f(pts) == exp, (name, pts, f(pts), exp)
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên giữa mọi cách (seed cố định, đúng ràng buộc)

Sinh điểm trên lưới nhỏ để có nhiều bộ ba thẳng hàng, `n` từ 1 tới 10; các cách độc lập nhau (tổ hợp đường, phân hoạch, DP, BFS) phải trùng kết quả.

In [20]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 9); R = rng.choice([2, 3, 5, 100])
    s = set()
    while len(s) < n: s.add((rng.randint(-R, R), rng.randint(-R, R)))
    pts = [list(p) for p in s]
    ref = backtrack_groups(pts)
    for name, f in ALL.items():
        assert f(pts) == ref, (name, pts)
print("400 case ngẫu nhiên OK")

400 case ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≤ ⌈n/2⌉; dịch/hoán vị điểm không đổi đáp án

In [21]:
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(1, 10); s = set()
    while len(s) < n: s.add((rng.randint(-4, 4), rng.randint(-4, 4)))
    pts = [list(p) for p in s]; a = Solution().minimumLines(pts)
    assert 1 <= a <= (n + 1) // 2
    dx, dy = rng.randint(-50, 50), rng.randint(-50, 50)
    moved = [[x + dx, y + dy] for x, y in pts]; rng.shuffle(moved)
    assert Solution().minimumLines(moved) == a
    swapped = [[y, x] for x, y in pts]                    # phản chiếu qua y = x
    assert Solution().minimumLines(swapped) == a
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn chọn đường phủ nhiều điểm **mới** nhất

In [22]:
def wrong_greedy(points):
    n = len(points); full = (1 << n) - 1
    L = sorted(line_masks(points)); cov = 0; cnt = 0
    while cov != full:
        best = max(L, key=lambda m: bin(m & ~cov).count("1"))
        cov |= best; cnt += 1
    return cnt

print("P_bad =", P_bad)
print("bản sai :", wrong_greedy(P_bad))
print("bản đúng:", Solution().minimumLines(P_bad))

P_bad = [[0, 2], [1, 0], [1, 1], [1, 3], [2, 0], [2, 1], [3, 1], [3, 2]]
bản sai : 4
bản đúng: 3


**Vì sao sai:** set cover tổng quát là NP-khó; tham lam chỉ cho xấp xỉ. Ở `P_bad`, chọn đường 3 điểm "hấp dẫn" đầu tiên lại phá vỡ cách ghép 3 đường tối ưu. **Sửa:** DP/vét cạn (n ≤ 10 cho phép).

### Lỗi 2 — Dùng hệ số góc số thực (chia cho 0 / sai số)

In [23]:
def wrong_slope_collinear(a, b, c):
    try:
        return (b[1] - a[1]) / (b[0] - a[0]) == (c[1] - a[1]) / (c[0] - a[0])
    except ZeroDivisionError:
        return "ZeroDivisionError"

print("đường đứng      :", wrong_slope_collinear([1, 0], [1, 5], [1, 9]), "| đúng:", collinear([1, 0], [1, 5], [1, 9]))
print("float sai số    :", wrong_slope_collinear([0, 0], [3, 1], [-99, -33]),
      "| hệ số góc 1/3 vs -33/-99:", 1 / 3, -33 / -99)

đường đứng      : ZeroDivisionError | đúng: True
float sai số    : True | hệ số góc 1/3 vs -33/-99: 0.3333333333333333 0.3333333333333333


Với tọa độ nhỏ như đề (|x|, |y| ≤ 100), phép chia float được **làm tròn đúng** nên hai phân số bằng nhau cho cùng một float — ví dụ `1/3` và `−33/−99` ở trên. Vấn đề thật sự là **chia cho 0** với đường thẳng đứng (bản sai phải xử lý riêng, dễ quên), và khi tọa độ lớn (≈10⁹) hai hệ số góc **khác nhau** có thể làm tròn thành cùng một float. Ví dụ với tọa độ lớn:

In [24]:
a, b, c = [0, 0], [10**9, 10**9 - 1], [10**9 - 1, 10**9 - 2]
print("float nói thẳng hàng?", wrong_slope_collinear(a, b, c), "| tích chéo (chính xác):", collinear(a, b, c))

float nói thẳng hàng? True | tích chéo (chính xác): False


**Sửa:** tích chéo số nguyên — luôn chính xác, không cần trường hợp riêng.

### Lỗi 3 — Mỗi đường chỉ phủ đúng hai điểm `{i, j}` (quên các điểm thẳng hàng khác)

In [25]:
def wrong_pair_only(points):
    n = len(points); full = (1 << n) - 1
    dp = [inf] * (1 << n); dp[0] = 0
    for mask in range(full):
        if dp[mask] == inf: continue
        i = (~mask & (mask + 1)).bit_length() - 1
        cands = [1 << i] + [(1 << i) | (1 << j) for j in range(n) if j != i]     # SAI: không mở rộng thành đường tối đa
        for L in cands:
            dp[mask | L] = min(dp[mask | L], dp[mask] + 1)
    return dp[full]

print("bản sai :", wrong_pair_only(examples[1][0]), "| đúng:", Solution().minimumLines(examples[1][0]))
print("bản sai :", wrong_pair_only(examples[0][0]), "| đúng:", Solution().minimumLines(examples[0][0]))

bản sai : 2 | đúng: 1
bản sai : 2 | đúng: 2


**Vì sao sai:** bản sai luôn cho `⌈n/2⌉` vì mỗi "đường" chỉ phủ 2 điểm. Ví dụ 2 có 3 điểm thẳng hàng ⇒ đúng là 1 đường. **Sửa:** mask của đường qua `i, j` phải gồm **mọi** `k` thẳng hàng với `i, j`.

Còn đây là một "nghi ngờ" hay gặp nhưng **không phải lỗi**: chỉ ghép `i` với các `j` **chưa phủ**. Ta kiểm chứng nó tương đương bản đầy đủ (lập luận ở mục 6):

In [26]:
def only_uncovered_j(points):
    n = len(points); full = (1 << n) - 1
    dp = [inf] * (1 << n); dp[0] = 0
    for mask in range(full):
        if dp[mask] == inf: continue
        i = (~mask & (mask + 1)).bit_length() - 1
        cands = [1 << i] + [sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                            for j in range(n) if j != i and not (mask >> j & 1)]
        for L in cands:
            dp[mask | L] = min(dp[mask | L], dp[mask] + 1)
    return dp[full]

rng = random.Random(3)
for _ in range(400):
    s = set()
    while len(s) < rng.randint(1, 9): s.add((rng.randint(0, 3), rng.randint(0, 3)))
    pts = [list(p) for p in s]
    assert only_uncovered_j(pts) == Solution().minimumLines(pts)
print("chỉ ghép với j chưa phủ: tương đương trên 400 mẫu (một tối ưu hợp lệ, không phải lỗi)")

chỉ ghép với j chưa phủ: tương đương trên 400 mẫu (một tối ưu hợp lệ, không phải lỗi)


### Lỗi 4 — Quên đường "chỉ một điểm" (khi điểm cuối cùng lẻ loi)

In [27]:
def wrong_no_single(points):
    n = len(points); full = (1 << n) - 1
    dp = [inf] * (1 << n); dp[0] = 0
    for mask in range(full):
        if dp[mask] == inf: continue
        i = (~mask & (mask + 1)).bit_length() - 1
        for j in range(n):
            if j != i:
                L = sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                dp[mask | L] = min(dp[mask | L], dp[mask] + 1)
    return dp[full]

print("một điểm  - bản sai:", wrong_no_single([[5, 5]]), "| đúng:", Solution().minimumLines([[5, 5]]))
print("ví dụ 1   - bản sai:", wrong_no_single(examples[0][0]), "| đúng:", Solution().minimumLines(examples[0][0]))

một điểm  - bản sai: inf | đúng: 1
ví dụ 1   - bản sai: 2 | đúng: 2


Với `n ≥ 2`, luôn có điểm `j ≠ i` nên bản sai ít khi lộ — nhưng với **một điểm** thì không có `j` nào ⇒ `inf`. **Sửa:** thêm `1 << i` vào danh sách ứng viên.

### Lỗi 5 — Đếm số **đường phân biệt** đi qua ≥ 2 điểm (hiểu nhầm đề)

In [28]:
def wrong_count_lines(points):
    return len({line_key(points[i], points[j]) for i in range(len(points)) for j in range(i + 1, len(points))})

print("bản sai :", wrong_count_lines(examples[0][0]))
print("bản đúng:", Solution().minimumLines(examples[0][0]))

bản sai : 4
bản đúng: 2


**Sửa:** đề hỏi **ít nhất** bao nhiêu đường để **phủ** mọi điểm, không phải có bao nhiêu đường khác nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 10–16` + "chọn tập con / phủ hết" ⇒ **bitmask DP** trên tập đã phủ, mẹo "xử lý phần tử chưa phủ nhỏ nhất" giảm số chuyển trạng thái từ O(số lựa chọn) xuống O(số lựa chọn chứa phần tử đó).
- Hình học với tọa độ nguyên ⇒ **tích chéo**, **gcd** để chuẩn hóa, tránh float.
- Một đường qua ≥ 2 điểm được xác định bởi một cặp ⇒ tập ứng viên O(n²).
- Set cover tổng quát: tham lam **không** tối ưu.
- Bài cùng họ: 149 Max Points on a Line, 1494 Parallel Courses II (bitmask DP), 1723, 2305 Fair Distribution of Cookies, 698 Partition to K Equal Sum Subsets.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **chính các đường** (mỗi đường là danh sách điểm nó phủ)

In [29]:
def lines_solution(points):
    n = len(points); full = (1 << n) - 1
    through = [{1 << i} | {sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                          for j in range(n) if j != i} for i in range(n)]
    dp = [inf] * (1 << n); dp[0] = 0; par = [None] * (1 << n)
    for mask in range(full):
        if dp[mask] == inf: continue
        i = (~mask & (mask + 1)).bit_length() - 1
        for L in through[i]:
            nm = mask | L
            if dp[mask] + 1 < dp[nm]: dp[nm] = dp[mask] + 1; par[nm] = (mask, L)
    out = []; m = full
    while m:
        pm, L = par[m]
        out.append([points[k] for k in range(n) if L >> k & 1 and not pm >> k & 1])
        m = pm
    return out[::-1]

for pts, _ in examples + [(P_bad, None)]:
    sol_lines = lines_solution(pts)
    print(len(sol_lines), "đường:", sol_lines)
    assert all(len(g) < 3 or all(collinear(g[0], g[1], p) for p in g) for g in sol_lines)

2 đường: [[[0, 1], [2, 3], [4, 5]], [[4, 3]]]
1 đường: [[[0, 2], [-2, -2], [1, 4]]]
3 đường: [[[0, 2], [1, 1], [2, 0]], [[1, 0], [2, 1], [3, 2]], [[1, 3], [3, 1]]]


### 15.2 Biến thể B — Chỉ được dùng đường **ngang hoặc dọc**

Mỗi điểm `(x, y)` phải được phủ bởi đường `X = x` hoặc `Y = y` ⇒ chọn ít nhất các "giá trị x" và "giá trị y" sao cho mọi điểm có x hoặc y được chọn = **phủ đỉnh nhỏ nhất** trên đồ thị hai phía (x ↔ y, mỗi điểm là một cạnh). Theo **định lý König**, bằng **cặp ghép cực đại** ⇒ thuật toán Kuhn O(V·E).

In [30]:
def axis_lines(points):
    xs = sorted({x for x, _ in points}); adj = {x: [] for x in xs}
    for x, y in points: adj[x].append(y)
    match_y = {}
    def try_kuhn(x, seen):
        for y in adj[x]:
            if y in seen: continue
            seen.add(y)
            if y not in match_y or try_kuhn(match_y[y], seen):
                match_y[y] = x; return True
        return False
    return sum(try_kuhn(x, set()) for x in xs)

def axis_lines_brute(points):
    xs = sorted({x for x, _ in points}); ys = sorted({y for _, y in points}); best = len(points)
    from itertools import product
    for cx in product([0, 1], repeat=len(xs)):
        X = {x for x, c in zip(xs, cx) if c}
        need_y = {y for x, y in points if x not in X}
        best = min(best, len(X) + len(need_y))
    return best

print(axis_lines(examples[0][0]), axis_lines([[x, y] for x in range(3) for y in range(3)]))
rng = random.Random(152)
for _ in range(400):
    s = set()
    while len(s) < rng.randint(1, 10): s.add((rng.randint(0, 4), rng.randint(0, 4)))
    pts = [list(p) for p in s]
    assert axis_lines(pts) == axis_lines_brute(pts)
print("König: khớp brute force")

3 3
König: khớp brute force


### 15.3 Biến thể C — LC 149: **số điểm nhiều nhất** trên một đường

In [31]:
def max_points_on_line(points):
    n = len(points)
    if n <= 2: return n
    best = 0
    for i in range(n):
        cnt = defaultdict(int)
        for j in range(n):
            if j != i: cnt[line_key(points[i], points[j])] += 1
        best = max(best, 1 + max(cnt.values()))
    return best

print(max_points_on_line([[1, 1], [2, 2], [3, 3]]), max_points_on_line([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))

3 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tổ hợp k đường | thử mọi bộ k đường ứng viên | O(C(L,k)·k) | O(L) | kiểm chứng, n rất nhỏ |
| 1b. Quay lui phân hoạch | gán điểm vào nhóm thẳng hàng | mũ, cắt nhánh | O(n) | tư duy phân hoạch |
| 2. DP bottom-up | `dp[mask]`, phủ điểm nhỏ nhất | O(n³ + 2ⁿ n) | O(2ⁿ) | **mặc định** |
| 3. DP top-down | như 2, đệ quy có nhớ | ≤ như 2 | O(2ⁿ) | chỉ thăm mask đạt được |
| 4. Khóa (a,b,c) + BFS | gom đường bằng dict | O(n² log C + 2ⁿ L) | O(2ⁿ) | tái dùng cho LC 149 |
| 5. k tăng dần | quay lui + cận | rất nhanh khi k nhỏ | O(n) | khi 2ⁿ quá lớn nhưng k nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumLines([[0,0],[1,1],[2,0],[3,1],[4,0]])` và `minimumLines([[0,0],[2,1],[4,2],[1,5]])`?

**Bài 2 (tính tay):** Với 5 điểm `(0,0),(1,1),(2,2),(0,2),(2,0)` (hai đường chéo), liệt kê các đường ứng viên đi qua điểm `(0,0)` dưới dạng tập điểm.

**Bài 3 (điền TODO):** Viết `bt3_todo(points)` trả về `through` — với mỗi `i`, **tập các bitmask** đường đi qua `i` (gồm cả đường một điểm).

**Bài 4 (tự viết lại):** Viết `bt4_todo(points)` cài DP top-down (cách 3) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(points, k)` trả về `True` nếu phủ được mọi điểm bằng **tối đa `k`** đường, dùng quay lui có cắt nhánh (không dùng mảng `dp` kích thước `2ⁿ`).

In [32]:
def bt3_todo(points):
    # TODO: through[i] = tập bitmask các đường qua điểm i
    pass

def bt4_todo(points):
    # TODO: DP top-down trên mask
    pass

def bt5_todo(points, k):
    # TODO: phủ được bằng <= k đường?
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: hàng `y = 0` có 3 điểm, `y = 1` có 2. Mức 2: hai đường ngang phủ hết. Mức 3: `2`; ví dụ hai: `(0,0),(2,1),(4,2)` thẳng hàng trên `y = x/2`, `(1,5)` riêng ⇒ `2`.

**Bài 2.** Mức 1: ghép `(0,0)` với từng điểm khác. Mức 2: với `(1,1)` hoặc `(2,2)` ⇒ đường chéo chính. Mức 3: các tập: `{(0,0),(1,1),(2,2)}`, `{(0,0),(0,2)}`, `{(0,0),(2,0)}`, `{(0,0)}`.

**Bài 3.** Mức 1: hai vòng lặp `i`, `j`. Mức 2: với mỗi cặp, OR các bit `k` thẳng hàng. Mức 3: thêm `1 << i`; dùng `set` để khử trùng.

**Bài 4.** Mức 1: `f(full) = 0`. Mức 2: `i` = bit 0 thấp nhất. Mức 3: `f(mask) = 1 + min(f(mask | L) for L in through[i])` với `@lru_cache`.

**Bài 5.** Mức 1: hàm `can(mask, r)`. Mức 2: hết đường mà chưa phủ ⇒ False. Mức 3: thử mọi `L` qua điểm chưa phủ nhỏ nhất; cận: `r · maxLine < số điểm còn lại` ⇒ False.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `2`. **Bài 2:** 4 tập như gợi ý mức 3.

In [33]:
assert Solution().minimumLines([[0, 0], [1, 1], [2, 0], [3, 1], [4, 0]]) == 2
assert Solution().minimumLines([[0, 0], [2, 1], [4, 2], [1, 5]]) == 2

def bt3(points):
    n = len(points)
    return [{1 << i} | {sum(1 << k for k in range(n) if collinear(points[i], points[j], points[k]))
                       for j in range(n) if j != i} for i in range(n)]

diag = [[0, 0], [1, 1], [2, 2], [0, 2], [2, 0]]
got = {frozenset(tuple(diag[k]) for k in range(5) if m >> k & 1) for m in bt3(diag)[0]}
assert got == {frozenset({(0, 0), (1, 1), (2, 2)}), frozenset({(0, 0), (0, 2)}), frozenset({(0, 0), (2, 0)}), frozenset({(0, 0)})}

def bt4(points):
    n = len(points); full = (1 << n) - 1; th = bt3(points)
    @lru_cache(None)
    def f(mask):
        if mask == full: return 0
        i = (~mask & (mask + 1)).bit_length() - 1
        return 1 + min(f(mask | L) for L in th[i])
    return f(0)

def bt5(points, k):
    n = len(points); full = (1 << n) - 1; th = bt3(points)
    mx = max(bin(m).count("1") for t in th for m in t)
    def can(mask, r):
        if mask == full: return True
        if r == 0 or r * mx < n - bin(mask).count("1"): return False
        i = (~mask & (mask + 1)).bit_length() - 1
        return any(can(mask | L, r - 1) for L in th[i])
    return can(0, k)

rng = random.Random(1952)
for _ in range(300):
    n = rng.randint(1, 9); s = set()
    while len(s) < n: s.add((rng.randint(-3, 3), rng.randint(-3, 3)))
    pts = [list(p) for p in s]; ref = backtrack_groups(pts)
    assert bt4(pts) == ref
    assert bt5(pts, ref) and (ref == 1 or not bt5(pts, ref - 1))
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def collinear(a, b, c):
    return (b[0]-a[0])*(c[1]-a[1]) == (c[0]-a[0])*(b[1]-a[1])
through[i] = {1<<i} ∪ {mask(line(i, j)) for j != i}      # mask = MỌI điểm thẳng hàng với i, j
dp = [inf] * (1<<n); dp[0] = 0
for mask in range(full):
    if dp[mask] == inf: continue
    i = (~mask & (mask+1)).bit_length() - 1            # điểm chưa phủ nhỏ nhất
    for L in through[i]:
        dp[mask|L] = min(dp[mask|L], dp[mask] + 1)
return dp[full]
```

**Checklist:**
- [ ] Thẳng hàng bằng **tích chéo số nguyên** (không float, không chia).
- [ ] Đường ứng viên là đường **tối đa** `line(i, j)` (mọi điểm thẳng hàng) + đường một điểm.
- [ ] Chỉ mở rộng từ điểm **chưa phủ nhỏ nhất**.
- [ ] `n = 1` ⇒ 1; hai điểm ⇒ 1.
- [ ] Không dùng tham lam.
- [ ] (Premium) Đối chiếu lại đề gốc.